# Experiment 9: Evaluate Trained Neural Network Performance

**Objective:** To evaluate the performance of a trained neural network using suitable evaluation metrics in Python.

## Theory

Training accuracy alone does not tell the full story. We use several metrics:

- **Accuracy** — overall, what fraction of predictions were correct?
- **Precision** — of everything predicted as class X, how many were actually X? (Are we over-predicting X?)
- **Recall** — of everything that IS actually class X, how many did we correctly find? (Are we missing X?)
- **F1-Score** — harmonic mean of Precision and Recall.
- **Confusion Matrix** — a table showing exactly which classes are confused with which.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)
from tensorflow import keras
from tensorflow.keras import layers

data = load_iris()
X, y = data.data, data.target
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## Build & Train a Model

In [2]:
model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(10, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(3, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=50, batch_size=8, verbose=0)
print("Training complete.")


Training complete.


## Get Predictions & Compute Metrics

In [3]:
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro")
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Overall Accuracy : {accuracy * 100:.2f}%")
print(f"Macro Precision  : {precision:.4f}")
print(f"Macro Recall     : {recall:.4f}")
print(f"Macro F1-Score   : {f1:.4f}")


Overall Accuracy : 96.67%
Macro Precision  : 0.9697
Macro Recall     : 0.9667
Macro F1-Score   : 0.9666


## Confusion Matrix & Classification Report

In [4]:
cm = confusion_matrix(y_test, y_pred)
header = " " * 16 + "".join(f"{name:>12}" for name in class_names)
print(header)
for i, row in enumerate(cm):
    row_label = f"Actual {class_names[i]:<9}"
    print(row_label + "".join(f"{val:>12}" for val in row))

print()
print(classification_report(y_test, y_pred, target_names=class_names))


                      setosa  versicolor   virginica
Actual setosa             10           0           0
Actual versicolor           0           9           1
Actual virginica           0           0          10

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


## Conclusion

Accuracy alone can hide weaknesses (e.g. a model that struggles with one specific class). Precision, Recall, F1-score, and the Confusion Matrix together give a complete picture of how the trained network is actually performing on each class.